In [1]:
import torch
from torchvision import transforms
import os
from PIL import Image
from swin_unet import swin_Unet
import numpy as np
import matplotlib.pyplot as plt  
import cv2
import torchvision

In [2]:
model = swin_Unet(img_size=896, num_classes=1)
checkpoint = torch.load("C:\\Users\\fa578s\\Desktop\\RiWIX\\checkpoints\\checkpoint_ok.pth.tar", weights_only=False)
#checkpoint = torch.load("D:\\river_width\\checkpoints\\checkpoint0119.pth", weights_only=False)
model.load_state_dict(checkpoint["model"])
model = model.to('cuda')
model.eval()

c:\Users\fa578s\AppData\Local\anaconda3\Lib\site-packages\torch\functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:3638.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


swin_Unet(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  )
  (encoder): SwinTransformerDown(
    (pos_drop): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0): BasicLayer(
        (blocks): ModuleList(
          (0): SwinBlock(
            (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
            (attn): WindowAttention(
              (qkv): Linear(in_features=96, out_features=288, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=96, out_features=96, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
              (softmax): Softmax(dim=-1)
            )
            (drop_path): Identity()
            (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=96, out_features=384, bias=True)
            

In [4]:
test_img = 'C:\\Users\\fa578s\\Desktop\\preprocessed\\GLH\\test\\img'
test_mask = 'C:\\Users\\fa578s\\Desktop\\preprocessed\\GLH\\test\\label'
images = sorted(os.listdir(test_img))
masks = sorted(os.listdir(test_mask))

In [6]:
def calculate_metric_percase(pred, gt):

    pred = pred.cpu().detach().numpy().astype(np.uint8)
    gt = gt.cpu().detach().numpy().astype(np.uint8)

    # pred = pred.astype(np.uint8)
    # gt = gt.astype(np.uint8)
    
    if pred.ndim == 3:
        pred = pred.reshape(-1)
        gt = gt.reshape(-1)

    intersection = np.logical_and(pred==1, gt==1).sum()
    union = np.logical_or(pred==1, gt==1).sum()
    total_pixels = gt.size
    correct_pixels = (pred == gt).sum()
    dice_denom = (pred==1).sum() + (gt==1).sum()

    dice_score = (2.0 * intersection) / (dice_denom + 1e-8) if dice_denom > 0 else 1.0
    pixel_acc = correct_pixels / total_pixels
    iou_score = intersection / (union + 1e-8) if union > 0 else 1.0
    # dice_score = (2* (pred*gt).sum())/ ((pred+gt).sum() + 1e-8)
    return dice_score, pixel_acc, iou_score




In [7]:
import cv2
from thop import profile, clever_format
data_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225]) 
])
acc = 0
folder = "D:\\river_width\\en_dec_pred"
model.eval()
dice = 0
acc = 0
iou = 0
balanced_accuracies = []

  



with torch.no_grad():
    img_path = os.path.join(test_img, images[0])
    image = Image.open(img_path).copy()
    image = data_transforms(image)
    image = image.to('cuda').unsqueeze(0)
    inp = torch.rand(16, 3, 896, 896).to('cuda')
    torch.cuda.reset_peak_memory_stats()
    macs, n_param = profile(model, inputs=(inp,))
    flops = macs * 2
    gflops = flops / 1e9
    m_param = n_param / 1e6

    # GPU memory usage
    allocated_mem = torch.cuda.memory_allocated() / 1024**2  # in MB
    max_allocated_mem = torch.cuda.max_memory_allocated() / 1024**2  # in MB
    print(gflops, m_param, max_allocated_mem)
    for i in range(len(images)):
        # Load and preprocess image
        img_path = os.path.join(test_img, images[i])
        image = Image.open(img_path).copy()
        image = data_transforms(image)
        image = image.to('cuda').unsqueeze(0)
      
    

        
        # Predict
        pred = torch.sigmoid(model(image))
        pred = pred.squeeze().cpu()
        pred_np = pred.numpy().astype(np.float32)  
        blurred = cv2.GaussianBlur(pred_np, (5,5), sigmaX=1)
        pred = (blurred>0.5).astype(np.float32)
        pred = torch.from_numpy(pred).unsqueeze(0)
        

        # Save prediction
        torchvision.utils.save_image(pred, f"{folder}\\pred_{i}.png")

        # Load ground truth mask
        mask_path = os.path.join(test_mask, masks[i])
        print(img_path, mask_path)
        mask = Image.open(mask_path).convert('L')
        mask = transforms.ToTensor()(mask).squeeze(1)
        mask = (mask > 0.5).float()
        dice_score, pixel_acc, iou_score = calculate_metric_percase(pred, mask)
        dice += dice_score
        acc += pixel_acc
        iou += iou_score

        # Class 0 (background) accuracy
        class_0_mask = (mask == 0)
        correct_0 = ((pred == 0) & class_0_mask).sum().item()
        total_0 = class_0_mask.sum().item()
        acc_0 = correct_0 / total_0 if total_0 > 0 else float('nan')

        # Class 1 (foreground) accuracy
        class_1_mask = (mask == 1)
        correct_1 = ((pred == 1) & class_1_mask).sum().item()
        total_1 = class_1_mask.sum().item()
        acc_1 = correct_1 / total_1 if total_1 > 0 else float('nan')

        # Mean (balanced) accuracy for this image
        valid_acc = [a for a in [acc_0, acc_1] if not torch.isnan(torch.tensor(a))]
        balanced_acc = sum(valid_acc) / len(valid_acc) if valid_acc else 0.0

        balanced_accuracies.append(balanced_acc)

# Final average balanced accuracy across dataset
final_acc = sum(balanced_accuracies) / len(balanced_accuracies)
print(f"Balanced Test Accuracy (Mean over images): {final_acc:.2f}%")
print(f'Mean dice score : {dice/len(images)}')
print(f'Mean accuracy : {acc/len(images)}')
print(f'Mean iou score : {iou/len(images)}')


[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register count_softmax() for <class 'torch.nn.modules.activation.Softmax'>.
2830.845825024 358.826592 17157.4375
C:\Users\fa578s\Desktop\preprocessed\GLH\test\img\06.jpg C:\Users\fa578s\Desktop\preprocessed\GLH\test\label\06.png
C:\Users\fa578s\Desktop\preprocessed\GLH\test\img\07.jpg C:\Users\fa578s\Desktop\preprocessed\GLH\test\label\07.png
C:\Users\fa578s\Desktop\preprocessed\GLH\test\img\102.jpg C:\Users\fa578s\Desktop\preprocessed\GLH\test\label\102.png
C:\Users\fa578s\Desktop\preprocessed\GLH\test\img\110.jpg C:\Users\fa578s\Desktop\preprocessed\GLH\test\label\110.png
C:\Users\fa578s\Desktop\preprocessed\GLH\test\img\113.jpg C:\Users\fa578s\Desktop\preproces

In [8]:
import torch.nn.functional as F
swin_Unet = "D:\\river_width\\pred"
pred_images = os.listdir(swin_Unet)
dice = 0
acc = 0
iou = 0
balanced_accuracies = []
for i in range(len(pred_images)):

    pred_path = os.path.join(swin_Unet, pred_images[i])
    pred = Image.open(pred_path).convert('L')
    pred = transforms.ToTensor()(pred)
    pred = (pred > 0.5).float()

    mask_path = os.path.join(test_mask, masks[i])
    mask = Image.open(mask_path).convert('L')
    mask = transforms.ToTensor()(mask).unsqueeze(0)

    # Resize mask to match pred
    mask = F.interpolate(mask, size=pred.shape[1:], mode='bilinear', align_corners=False)
    mask = (mask > 0.5).float().squeeze(0)
    dice_score, pixel_acc, iou_score = calculate_metric_percase(pred, mask)
    dice += dice_score
    acc += pixel_acc
    iou += iou_score

    # Class 0 (background) accuracy
    class_0_mask = (mask == 0)
    correct_0 = ((pred == 0) & class_0_mask).sum().item()
    total_0 = class_0_mask.sum().item()
    acc_0 = correct_0 / total_0 if total_0 > 0 else float('nan')

    # Class 1 (foreground) accuracy
    class_1_mask = (mask == 1)
    correct_1 = ((pred == 1) & class_1_mask).sum().item()
    total_1 = class_1_mask.sum().item()
    acc_1 = correct_1 / total_1 if total_1 > 0 else float('nan')

    # Mean (balanced) accuracy for this image
    valid_acc = [a for a in [acc_0, acc_1] if not torch.isnan(torch.tensor(a))]
    balanced_acc = sum(valid_acc) / len(valid_acc) if valid_acc else 0.0

    balanced_accuracies.append(balanced_acc)

# Final average balanced accuracy across dataset
final_acc = sum(balanced_accuracies) / len(balanced_accuracies)
print(f"Balanced Test Accuracy (Mean over images): {final_acc:.2f}%")
    
print(f'Mean dice score : {dice/len(pred_images)}')
print(f'Mean accuracy : {acc/len(pred_images)}')
print(f'Mean iou score : {iou/len(pred_images)}')


Balanced Test Accuracy (Mean over images): 0.76%
Mean dice score : 0.5480825169511092
Mean accuracy : 0.8767203942123725
Mean iou score : 0.4445599915603311
